# W7｜教师演示：报告为什么给出两个相同的均值？

两组测量值分别是 `[1, 5, 12]` 和 `[2, 4]`。报告应给出两个均值 `(6.0, 3.0)`，实际却给出 `(6.0, 6.0)`。

**先猜一下。是均值公式算错了，还是报告传进去的数据错了？**

## 1. 复现

按顺序普通运行下面三格。先看直接调用，再看报告。

In [ ]:
# 定义
import numpy as np

def mean(values):
    return float(np.mean(values))


def report(group_a, group_b):
    mean_a = mean(group_a)
    mean_b = mean(group_a)
    return mean_a, mean_b


In [ ]:
# 直接调用
group_a = np.array([1., 5., 12.])
group_b = np.array([2., 4.])

print("两组 shape", group_a.shape, group_b.shape)
print("直接调用", (mean(group_a), mean(group_b)))


In [ ]:
# 报告
print("报告", report(group_a, group_b))

## 2. 用断点追踪两次调用

### 浏览器 Jupyter

1. 启用顶部工具栏的小虫图标；如提示重启 kernel，先确认。随后按顺序普通运行「定义」和「直接调用」两格，再点 **View → Debugger Panel** 打开右侧调试面板。
2. 在「定义」格的 `return float(np.mean(values))` 行号左侧点出红色断点。选中「报告」格，点击顶部的 **▶ 运行**。两次调用会停在同一个断点。
3. **第一次停住：** 在右侧 **VARIABLES** 查看 `values`，有 3 项，`shape` 是 `(3,)`。先预测：如果第二次传的是第二组，`values.shape` 应该是多少？
4. 点 **CALLSTACK** 区域的 **Continue**。**第二次停住：** 对照刚才预测的 `(2,)`，观察 `values` 实际仍有 3 项，`shape` 仍是 `(3,)`。形状相同不能证明是同一个数组对象。
5. 在 **CALLSTACK** 中选择上一层 `report`，看高亮的是哪一行、这一行实际传入了什么。请用一句话说明：**预期输入 → 实际输入 → 调用位置**。
6. 点 **Continue** 让「报告」格运行完，移除断点，再进入修改与检查。

<details>
<summary>在 VS Code 中操作时</summary>

在「定义」格的 `return` 行设断点，对「报告」格选择 **Debug Cell**。在 **Variables** 看 `values`，用 **Continue** 到第二次调用，再从 **Call Stack** 选择 `report` 核对实参。

</details>

## 3. 改一处，重跑检查

根据刚才定位的调用行，只改调用者 `report` 的一处；`mean` 的计算保持原样。普通运行「定义」→「直接调用」→「报告」→「检查」，核对原用例和换过输入的第二个用例。

In [ ]:
# 检查
np.testing.assert_allclose(report(group_a, group_b), (6.0, 3.0))
np.testing.assert_allclose(
    report(np.array([2., 4.]), np.array([10., 20.])),
    (3.0, 15.0),
)
print("两个用例均通过")


<details>
<summary>演示备用记录（点击展开）</summary>

普通运行时，直接调用得到 `(6.0, 3.0)`，原报告得到 `(6.0, 6.0)`。

两次停在 `mean` 内，变量 `values` 都是 `[1, 5, 12]`。切到第二次的调用者 `report`，看到 `mean_b = mean(group_a)`。

改成 `mean_b = mean(group_b)` 后，原用例得到 `(6.0, 3.0)`，换输入的用例得到 `(3.0, 15.0)`。

</details>